# 03. Training (Text Cell)

ADR-016 §3-2 / ADR-017 §3-2 모델 5종 × polluter 5종 × level 6 × dataset 6 학습.

분류 트랙: LogReg+TFIDF / TextCNN / DistilBERT / BERT-base / RoBERTa-base
회귀 트랙: Ridge+TFIDF / XGBoost+TFIDF / TextCNN-Reg / DistilBERT-Reg / BERT-base-Reg

총 학습 건수: 6 dataset × 5 model × 5 polluter × 6 level + 6×5 baseline = 930건
T4 기준 transformer ≈ 5~20분 → 30~50시간 (분류) + 30~50시간 (회귀)

Output: `results/text_train_metrics.csv` (rows = dataset × model × polluter × level × seed)

---


## 0. import + 모델 정의


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import sys, os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, r2_score

BASE = '/content/drive/MyDrive/capstone/dsc'
if BASE not in sys.path:
    sys.path.insert(0, BASE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


## 1. 모델 학습 함수

각 모델은 (train_texts, train_labels, test_texts, test_labels) → metric 반환.


In [ ]:
def train_logreg_tfidf(tr_t, tr_y, te_t, te_y, max_features=20000):
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    Xtr = vec.fit_transform(tr_t); Xte = vec.transform(te_t)
    clf = LogisticRegression(max_iter=2000, n_jobs=-1, random_state=42).fit(Xtr, tr_y)
    return accuracy_score(te_y, clf.predict(Xte))


def train_ridge_tfidf(tr_t, tr_y, te_t, te_y, max_features=20000, alpha=1.0):
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    Xtr = vec.fit_transform(tr_t); Xte = vec.transform(te_t)
    clf = Ridge(alpha=alpha, random_state=42).fit(Xtr, tr_y)
    return r2_score(te_y, clf.predict(Xte))


def train_xgb_tfidf(tr_t, tr_y, te_t, te_y, max_features=20000):
    from xgboost import XGBRegressor
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    Xtr = vec.fit_transform(tr_t); Xte = vec.transform(te_t)
    clf = XGBRegressor(max_depth=6, n_estimators=500, learning_rate=0.05,
                        random_state=42, n_jobs=-1).fit(Xtr, tr_y)
    return r2_score(te_y, clf.predict(Xte))


In [ ]:
# ============================================================
# TextCNN — 분류/회귀 head 교체 (ADR-016/017 §3-2 사전등록)
# ============================================================
class _SimpleTokenizer:
    """random init embedding용 whitespace 토큰화 + 단순 vocab.

    DistilBERT BPE와 다르지만 TextCNN baseline에서는 충분.
    UNK=1, PAD=0 고정. max_len truncation.
    """
    PAD, UNK = 0, 1

    def __init__(self, max_vocab=20000):
        self.max_vocab = max_vocab
        self.itos = ['[PAD]', '[UNK]']
        self.stoi = {'[PAD]': 0, '[UNK]': 1}

    def fit(self, texts):
        from collections import Counter
        cnt = Counter()
        for t in texts:
            cnt.update(t.split())
        for tok, _ in cnt.most_common(self.max_vocab - 2):
            self.stoi[tok] = len(self.itos)
            self.itos.append(tok)
        return self

    def encode(self, text, max_len):
        ids = [self.stoi.get(tok, self.UNK) for tok in text.split()[:max_len]]
        if len(ids) < max_len:
            ids = ids + [self.PAD] * (max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.itos)


class TextCNN(nn.Module):
    def __init__(self, vocab_size, n_class, emb=128, kernels=(3, 4, 5), filters=100,
                 dropout=0.5, regression=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb, filters, k, padding=k // 2) for k in kernels])
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(filters * len(kernels), 1 if regression else n_class)
        self.regression = regression

    def forward(self, x):
        x = self.emb(x).transpose(1, 2)
        x = torch.cat([torch.max(torch.relu(c(x)), dim=2).values for c in self.convs], dim=1)
        x = self.drop(x)
        out = self.fc(x)
        return out.squeeze(-1) if self.regression else out


def train_textcnn(tr_t, tr_y, te_t, te_y, regression=False, epochs=10,
                  batch=64, lr=1e-3, max_len=256, emb=128, filters=100,
                  device=None):
    """TextCNN finetune. 분류 → accuracy, 회귀 → R²."""
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    tok = _SimpleTokenizer().fit(tr_t)

    def to_tensor(texts, ys):
        X = torch.tensor([tok.encode(t, max_len) for t in texts], dtype=torch.long)
        if regression:
            y = torch.tensor(list(ys), dtype=torch.float32)
        else:
            y = torch.tensor(list(ys), dtype=torch.long)
        return X, y

    Xtr, ytr = to_tensor(tr_t, tr_y)
    Xte, yte = to_tensor(te_t, te_y)

    n_class = 1 if regression else int(max(int(max(tr_y)), int(max(te_y))) + 1)
    model = TextCNN(len(tok), n_class, emb=emb, filters=filters, regression=regression).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss() if regression else nn.CrossEntropyLoss()

    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        preds = []
        for i in range(0, len(Xte), batch):
            xb = Xte[i:i + batch].to(device)
            out = model(xb)
            if regression:
                preds.append(out.cpu().numpy())
            else:
                preds.append(out.argmax(dim=1).cpu().numpy())
        preds = np.concatenate(preds)

    if regression:
        return float(r2_score(yte.numpy(), preds))
    return float(accuracy_score(yte.numpy(), preds))


In [ ]:
# ============================================================
# Transformer (DistilBERT/BERT/RoBERTa) — head 교체로 분류/회귀 둘 다 처리
# ADR-016/017 §3-2 사전등록 (max_len=256, batch=32, epoch=3, lr=2e-5, AdamW)
# ============================================================
def train_transformer(model_id, tr_t, tr_y, te_t, te_y, regression=False,
                      max_len=256, batch=32, epochs=3, lr=2e-5,
                      weight_decay=0.01, output_dir=None):
    """HuggingFace AutoModelForSequenceClassification finetune."""
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        Trainer, TrainingArguments, DataCollatorWithPadding,
    )
    from datasets import Dataset

    n_label = 1 if regression else int(max(int(max(tr_y)), int(max(te_y))) + 1)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=n_label,
        problem_type='regression' if regression else 'single_label_classification')

    def to_hf(texts, ys):
        return Dataset.from_dict({
            'text': list(texts),
            'labels': [float(y) for y in ys] if regression else [int(y) for y in ys],
        })

    def tokenize(batch):
        return tok(batch['text'], truncation=True, max_length=max_len)

    ds_tr = to_hf(tr_t, tr_y).map(tokenize, batched=True, remove_columns=['text'])
    ds_te = to_hf(te_t, te_y).map(tokenize, batched=True, remove_columns=['text'])

    args = TrainingArguments(
        output_dir=output_dir or f'./_tmp_{os.getpid()}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch,
        learning_rate=lr,
        weight_decay=weight_decay,
        logging_strategy='no',
        save_strategy='no',
        report_to='none',
        seed=42,
    )

    def metric_fn(eval_pred):
        preds, labels = eval_pred
        if regression:
            preds = preds.squeeze(-1) if preds.ndim > 1 else preds
            return {'r2': float(r2_score(labels, preds))}
        return {'accuracy': float(accuracy_score(labels, preds.argmax(axis=-1)))}

    trainer = Trainer(
        model=model, args=args,
        train_dataset=ds_tr, eval_dataset=ds_te,
        tokenizer=tok,
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=metric_fn,
    )
    trainer.train()
    metrics = trainer.evaluate()
    return metrics.get('eval_r2' if regression else 'eval_accuracy', float('nan'))


## 2. 학습 스윕

02의 sweep 결과(polluted texts/labels)를 직접 메모리에서 사용하거나 ↓ 처럼 csv에서 재구성.


In [ ]:
# 학습 루프 — Colab GPU 환경에서 큐로 백그라운드 실행
# results = []
# for ds_name, (tr_texts, tr_y, te_texts, te_y) in datasets.items():
#     for pol_name, pol_cls in CLASSIFICATION_POLLUTERS.items():
#         for lvl in LEVEL_GRID:
#             pol = pol_cls(lvl, random_seed=42)
#             tr_p, tr_yp = pol.pollute(tr_texts, tr_y)
#             for model_name, train_fn in MODELS.items():
#                 metric = train_fn(tr_p, tr_yp, te_texts, te_y)
#                 results.append({...})
# pd.DataFrame(results).to_csv('results/text_train_metrics.csv', index=False)


---

다음: `04_scoreboard_text.ipynb` — r 분석, polluter hold-out, default vs tuned 가중치.
